# AO3 Metadata Scraper

Scrapes **metadata only** (no fic body text) from [Archive of Our Own](https://archiveofourown.org).

Inspired by [radiolarian/AO3Scraper](https://github.com/radiolarian/AO3Scraper).

## Three explicit steps

| Step | What it does | Output file |
|---|---|---|
| **1 — Build page list** | Generates every listing-page URL for your search | `ao3_pages.txt` |
| **2 — Collect work IDs** | Fetches each listing page and extracts work IDs | `ao3_work_ids.txt` |
| **3 — Collect metadata** | Fetches each work page and writes metadata to CSV | `ao3_metadata.csv` |

Each step saves its output to a plain file you can inspect before the next step runs.
Each step is resumable if interrupted.

## Output columns
`work_id` · `title` · `author` · `rating` · `warnings` · `category` · `fandom` ·
`relationship` · `character` · `additional_tags` · `language` · `series` ·
`published` · `status` · `status_date` · `words` · `chapters` · `comments` ·
`kudos` · `bookmarks` · `hits` · `summary`

## ⚠ Rate limiting
AO3's terms of service require a **minimum 5-second delay** between every request.
Do not reduce `REQUEST_DELAY` below 5.  See: https://archiveofourown.org/TOS


---
## Step 1 — Install dependencies

In [ ]:
%pip install -q requests beautifulsoup4
print("Ready.")


---
## Step 2 — Configuration

**This is the only cell you need to edit.**

### How to find your URL
1. Go to [archiveofourown.org](https://archiveofourown.org)
2. Browse to a fandom/tag page or run a search
3. Apply filters as desired (language, rating, completion, word count, etc.)
4. Copy the full URL from your browser's address bar

### `NUM_PAGES`
- Set to an integer (e.g. `5`) to scrape exactly that many pages, with no network call needed in Step 3.
- Leave as `None` to auto-detect from the pagination bar on page 1.
- AO3 shows 20 works per page, so `NUM_PAGES = 5` gives up to 100 works.


In [ ]:
# ── ✏️  Edit these ─────────────────────────────────────────────────────────────
SEARCH_URL  = "https://archiveofourown.org/works?work_search[sort_column]=kudos_count&tag_id=Sherlock+%28TV%29"
START_PAGE  = 1    # first page to scrape (inclusive)
END_PAGE    = 5    # last page to scrape (inclusive)
PAGES_FILE  = "ao3_pages.txt"
IDS_FILE    = "ao3_work_ids.txt"
OUTPUT_FILE = "ao3_metadata.csv"
RESUME      = False         # set True to continue an interrupted Step 6
USER_HEADER = ""            # optional: "MyProject/1.0; email@example.com"
# ──────────────────────────────────────────────────────────────────────────────

REQUEST_DELAY   = 5.0   # seconds between every request — AO3 ToS minimum
MAX_RETRIES     = 5      # attempts per URL before giving up
RETRY_DELAY     = 15.0   # initial back-off in seconds (doubles each retry)
REQUEST_TIMEOUT = 60     # seconds per request — AO3 can be slow
_RETRY_STATUSES = {429, 500, 502, 503, 504, 525}  # transient errors worth retrying
AO3_BASE      = "https://archiveofourown.org"

METADATA_COLUMNS = [
    "work_id", "title", "author", "rating", "warnings", "category",
    "fandom", "relationship", "character", "additional_tags", "language",
    "series", "published", "status", "status_date", "words", "chapters",
    "comments", "kudos", "bookmarks", "hits", "summary",
]

print(f"URL        : {SEARCH_URL}")
print(f"Pages      : {START_PAGE} → {END_PAGE}  ({END_PAGE - START_PAGE + 1} pages)")
print(f"Pages file : {PAGES_FILE}")
print(f"IDs file   : {IDS_FILE}")
print(f"Output     : {OUTPUT_FILE}")


---
## Step 3 — Imports and HTTP helper

`fetch()` is the only function that makes network requests. It enforces the
mandatory AO3 delay, retries on failures, and backs off longer on 429 responses.


In [ ]:
import csv, os, re, time
from pathlib import Path
from urllib.parse import urlencode, urlparse, parse_qs, urlunparse

import requests
from bs4 import BeautifulSoup


def make_session(user_header=""):
    """Return a requests.Session with a descriptive User-Agent."""
    session = requests.Session()
    ua = "Mozilla/5.0 (compatible; ao3-metadata-scraper/1.0)"
    if user_header:
        ua = f"{ua}; {user_header}"
    session.headers.update({"User-Agent": ua})
    return session


def fetch(url, session):
    """
    GET url, sleeping REQUEST_DELAY seconds first (AO3 ToS compliance).

    Retries up to MAX_RETRIES times with exponential back-off on any status
    in _RETRY_STATUSES or a network/timeout exception.

    Back-off schedule (RETRY_DELAY=15s): 15s -> 30s -> 60s -> 120s -> give up

    HTTP 525 (Cloudflare SSL handshake error) is treated as transient —
    it usually clears after one back-off wait.
    Timeouts use REQUEST_TIMEOUT (60s) and are also retried with back-off.
    HTTP 403 is NOT retried — it is a deliberate block, not a transient error.
    """
    time.sleep(REQUEST_DELAY)

    wait = RETRY_DELAY
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)

            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "html.parser")

            if resp.status_code in _RETRY_STATUSES:
                print(f"  HTTP {resp.status_code} "
                      f"(attempt {attempt}/{MAX_RETRIES}) — "
                      f"waiting {wait:.0f}s ...", flush=True)
                time.sleep(wait)
                wait *= 2       # exponential back-off: 15 -> 30 -> 60 -> 120s
                continue

            # Non-retryable (404, 403, etc.) — stop immediately
            print(f"  HTTP {resp.status_code}: {url}", flush=True)
            return None

        except requests.exceptions.Timeout:
            print(f"  Timed out after {REQUEST_TIMEOUT}s "
                  f"(attempt {attempt}/{MAX_RETRIES}) — "
                  f"waiting {wait:.0f}s ...", flush=True)
            time.sleep(wait)
            wait *= 2

        except requests.RequestException as exc:
            print(f"  Network error (attempt {attempt}/{MAX_RETRIES}): {exc} — "
                  f"waiting {wait:.0f}s ...", flush=True)
            time.sleep(wait)
            wait *= 2

    print(f"  Gave up after {MAX_RETRIES} attempts: {url}", flush=True)
    return None


SESSION = make_session(USER_HEADER)
print("Session ready.")


---
## Step 4 — Run Step 1: Build the page URL list

Set `START_PAGE` and `END_PAGE` in Step 2, then run this cell.

`_page_url()` injects `page=N` into the query string of `SEARCH_URL` and
increments N by 1 for each step from `START_PAGE` to `END_PAGE` (inclusive).
No network request is made — the full URL list is built immediately from
the values you provide.

**After this cell**, open `PAGES_FILE` and confirm the URLs look right before
moving to Step 5.

> **Tip — how many pages does your search have?**
> Open your search URL in a browser, scroll to the bottom, and read the last
> page number from the pagination bar.  Enter that as `END_PAGE`.
> AO3 shows 20 works per listing page, so 5 pages → ~100 works.


In [ ]:
def _page_url(base_url, page):
    """
    Return base_url with 'page=N' set in the query string.

    Replaces any existing page= parameter so calling _page_url on an
    already-paginated URL always gives the correct result.
    """
    from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
    parsed     = urlparse(base_url)
    qs         = parse_qs(parsed.query, keep_blank_values=True)
    qs["page"] = [str(page)]
    return urlunparse(parsed._replace(
        query=urlencode({k: v[0] for k, v in qs.items()})
    ))


def step1_build_page_list(base_url, start_page, end_page, pages_out):
    """
    Build every listing-page URL from start_page to end_page (inclusive)
    by incrementing the page number by 1 each step, and write to pages_out.

    No network requests are made — URLs are constructed directly from the
    values you supply.

    Parameters
    ----------
    base_url   : your AO3 search/tag/fandom URL (any page number in it is replaced)
    start_page : first page number to include (usually 1)
    end_page   : last page number to include
    pages_out  : file path to write the URL list (one URL per line)

    Returns
    -------
    list[str] — the ordered list of page URLs
    """
    if end_page < start_page:
        raise ValueError(f"end_page ({end_page}) must be >= start_page ({start_page})")

    page_urls = []
    page = start_page
    while page <= end_page:
        page_urls.append(_page_url(base_url, page))
        page += 1           # n+1 — advance to the next page

    from pathlib import Path
    Path(pages_out).write_text("\n".join(page_urls) + "\n", encoding="utf-8")
    print(f"  ✓ {len(page_urls)} URL(s) written to {pages_out}", flush=True)
    return page_urls


# ── Run Step 1 ────────────────────────────────────────────────────────────────
print(f"Step 1: building page URL list  (pages {START_PAGE} → {END_PAGE})\n")
PAGE_URLS = step1_build_page_list(SEARCH_URL, START_PAGE, END_PAGE, PAGES_FILE)

print(f"\nURLs written to {PAGES_FILE}:")
for url in PAGE_URLS:
    print(f"  {url}")


---
## Step 5 — Run Step 2: Collect work IDs

Fetches each listing page URL from `PAGES_FILE` and extracts the work IDs.

AO3 renders each work in a listing as `<li id="work_NNNNNNN" ...>`.
The numeric ID is taken directly from that attribute.

**After this cell**, open `IDS_FILE` — it should contain one numeric ID per line.
The count should be roughly 20 × the number of pages.


In [ ]:
def _ids_from_soup(soup):
    """Extract all work IDs from one listing page."""
    ids = []
    for li in soup.find_all("li", id=re.compile(r"^work_\d+")):
        raw = re.sub(r"^work_", "", li.get("id", "")).split()[0]
        if raw.isdigit():
            ids.append(raw)
    return ids


def step2_collect_ids(page_urls, session, ids_out):
    """
    Fetch every URL in page_urls, extract work IDs, write to ids_out.
    Deduplicates IDs that appear on multiple pages.
    """
    total  = len(page_urls)
    w      = len(str(total))
    all_ids = []

    for i, url in enumerate(page_urls, start=1):
        print(f"  [{i:>{w}}/{total}] {url}", flush=True)
        soup = fetch(url, session)
        if soup is None:
            print(f"    fetch failed — skipping", flush=True)
            continue
        page_ids = _ids_from_soup(soup)
        all_ids.extend(page_ids)
        print(f"    {len(page_ids)} IDs  (running total: {len(all_ids)})", flush=True)

    # Deduplicate, preserving order
    seen, unique = set(), []
    for wid in all_ids:
        if wid not in seen:
            seen.add(wid)
            unique.append(wid)

    if len(unique) < len(all_ids):
        print(f"  Deduplicated: {len(all_ids)} -> {len(unique)}", flush=True)

    Path(ids_out).write_text("\n".join(unique) + "\n", encoding="utf-8")
    print(f"\n  ✓ {len(unique)} work IDs written to {ids_out}", flush=True)
    return unique


# ── Run Step 2 ────────────────────────────────────────────────────────────────
# Re-load page URLs from file in case this cell runs independently
if "PAGE_URLS" not in dir() or not PAGE_URLS:
    PAGE_URLS = [ln.strip() for ln in Path(PAGES_FILE).read_text().splitlines()
                 if ln.strip()]

print(f"Step 2: collecting work IDs from {len(PAGE_URLS)} page(s)\n")
WORK_IDS = step2_collect_ids(PAGE_URLS, SESSION, IDS_FILE)
print(f"\nFirst 10 IDs: {WORK_IDS[:10]}")


---
## Step 6 — Run Step 3: Collect metadata

Fetches each individual work page and extracts metadata.

Each row is written and flushed immediately so progress is safe even if the
kernel is interrupted. Set `RESUME = True` in Step 2 and re-run to continue
from where it left off.

Works that are locked, deleted, or return Access Denied are recorded in
`errors_<OUTPUT_FILE>` — nothing is silently dropped.


In [ ]:
def _text(tag):
    return tag.get_text(strip=True) if tag else ""

def _tags(meta_dl, cls):
    dd = meta_dl.find("dd", class_=cls) if meta_dl else None
    return ", ".join(a.get_text(strip=True) for a in dd.find_all("a")) if dd else ""

def _stat(meta_dl, label):
    stats = meta_dl.find("dl", class_="stats") if meta_dl else None
    if not stats:
        return ""
    dt = stats.find("dt", string=re.compile(re.escape(label), re.I))
    return _text(dt.find_next_sibling("dd")) if dt else ""


def scrape_one(work_id, session):
    """
    Fetch one AO3 work page and return a metadata dict.
    Body text is never downloaded. Returns None on failure or Access Denied.
    Uses ?view_adult=true to bypass the mature-content gate.
    """
    soup = fetch(f"{AO3_BASE}/works/{work_id}?view_adult=true", session)
    if soup is None:
        return None
    if soup.find("p", string=re.compile(r"Access Denied", re.I)):
        print(f"  [work {work_id}] Access Denied", flush=True)
        return None

    title  = _text(soup.find("h2", class_="title heading"))
    byline = soup.find("h3", class_="byline heading")
    if byline:
        authors = [a.get_text(strip=True) for a in byline.find_all("a", rel="author")]
        author  = ", ".join(authors) if authors else _text(byline)
    else:
        author = ""

    sdiv    = soup.find("div", class_="summary module")
    bq      = sdiv.find("blockquote", class_="userstuff") if sdiv else None
    summary = bq.get_text(separator="\n", strip=True) if bq else ""

    meta = soup.find("dl", class_="work meta group")
    rating          = _tags(meta, "rating")
    warnings        = _tags(meta, "warning")
    category        = _tags(meta, "category")
    fandom          = _tags(meta, "fandom")
    relationship    = _tags(meta, "relationship")
    character       = _tags(meta, "character")
    additional_tags = _tags(meta, "freeform")
    language        = _text(meta.find("dd", class_="language") if meta else None)

    s_dd = meta.find("dd", class_="series") if meta else None
    if s_dd:
        parts  = [s.get_text(strip=True) for s in s_dd.find_all("span", class_="position")]
        series = "; ".join(parts) if parts else _text(s_dd)
    else:
        series = ""

    published = _stat(meta, "Published:")
    words     = _stat(meta, "Words:")
    chapters  = _stat(meta, "Chapters:")
    comments  = _stat(meta, "Comments:")
    kudos     = _stat(meta, "Kudos:")
    bookmarks = _stat(meta, "Bookmarks:")
    hits      = _stat(meta, "Hits:")

    c_dt = meta.find("dt", string=re.compile(r"Completed", re.I)) if meta else None
    u_dt = meta.find("dt", string=re.compile(r"Updated",   re.I)) if meta else None
    if c_dt:
        status, status_date = "Completed", _text(c_dt.find_next_sibling("dd"))
    elif u_dt:
        status, status_date = "Updated",   _text(u_dt.find_next_sibling("dd"))
    else:
        status = status_date = ""

    return {
        "work_id": work_id, "title": title, "author": author,
        "rating": rating, "warnings": warnings, "category": category,
        "fandom": fandom, "relationship": relationship, "character": character,
        "additional_tags": additional_tags, "language": language,
        "series": series, "published": published, "status": status,
        "status_date": status_date, "words": words, "chapters": chapters,
        "comments": comments, "kudos": kudos, "bookmarks": bookmarks,
        "hits": hits, "summary": summary,
    }


def step3_collect_metadata(work_ids, session, out, resume=False):
    """
    Scrape metadata for every work ID and stream rows to out (CSV).
    Flushes after every row. With resume=True, skips already-done IDs.
    """
    done = set()
    if resume and os.path.exists(out):
        with open(out, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                wid = row.get("work_id", "").strip()
                if wid:
                    done.add(wid)
        print(f"  Resuming — {len(done)} already done in {out}", flush=True)
        fout   = open(out, "a", newline="", encoding="utf-8")
        writer = csv.DictWriter(fout, fieldnames=METADATA_COLUMNS, extrasaction="ignore")
    else:
        fout   = open(out, "w", newline="", encoding="utf-8")
        writer = csv.DictWriter(fout, fieldnames=METADATA_COLUMNS, extrasaction="ignore")
        writer.writeheader()

    err_path = "errors_" + os.path.basename(out)
    ferr     = open(err_path, "a" if resume else "w", newline="", encoding="utf-8")
    err_w    = csv.writer(ferr)
    if not resume:
        err_w.writerow(["work_id", "reason"])

    todo    = [wid for wid in work_ids if wid not in done]
    total   = len(todo)
    w       = len(str(total))
    written = []

    try:
        for i, wid in enumerate(todo, start=1):
            print(f"  [{i:>{w}}/{total}] {wid} ... ", end="", flush=True)
            row = scrape_one(wid, session)
            if row:
                writer.writerow(row)
                fout.flush()
                written.append(row)
                print(f"✓  {row['title'][:55]}", flush=True)
            else:
                err_w.writerow([wid, "failed or access denied"])
                ferr.flush()
                print("✗  (logged)", flush=True)
    finally:
        fout.close()
        ferr.close()

    print(f"\n✓ {len(written)} rows written to {out}", flush=True)
    if os.path.exists(err_path) and os.path.getsize(err_path) > 20:
        print(f"  Errors logged to {err_path}", flush=True)
    return written


# ── Run Step 3 ────────────────────────────────────────────────────────────────
# Re-load IDs from file in case this cell runs independently
if "WORK_IDS" not in dir() or not WORK_IDS:
    WORK_IDS = [ln.strip() for ln in Path(IDS_FILE).read_text().splitlines()
                if ln.strip().isdigit()]
    print(f"Loaded {len(WORK_IDS)} IDs from {IDS_FILE}")

print(f"Step 3: scraping metadata for {len(WORK_IDS)} work(s)\n")
RESULTS = step3_collect_metadata(WORK_IDS, SESSION, OUTPUT_FILE, resume=RESUME)


---
## Step 7 — Preview results


In [ ]:
try:
    import pandas as pd
    df = pd.read_csv(OUTPUT_FILE)
    for col in ["words", "comments", "kudos", "bookmarks", "hits"]:
        df[col] = pd.to_numeric(df[col].str.replace(",", ""), errors="coerce")
    print(f"Shape: {df.shape[0]} works x {df.shape[1]} columns\n")
    df.head(10)
except FileNotFoundError:
    print(f"{OUTPUT_FILE} not found — run Steps 4–6 first.")
except ImportError:
    print("Run: %pip install pandas")


### Quick stats

In [ ]:
try:
    if 'df' in dir() and not df.empty:
        print("── Top 10 by kudos ───────────────────────────────────────────────")
        print(df[["title","author","fandom","kudos","words","status"]]
              .sort_values("kudos", ascending=False).head(10).to_string(index=False))
        print("\n── Rating distribution ───────────────────────────────────────────")
        print(df["rating"].value_counts().to_string())
        print("\n── Completion status ─────────────────────────────────────────────")
        print(df["status"].value_counts().to_string())
        print("\n── Word count summary ────────────────────────────────────────────")
        print(df["words"].describe().to_string())
except Exception as e:
    print(f"Error: {e}")
